# SahayakLLM — Notebook 01: Data, Metrics, Prompt Choice and Baselines (B0 / B1)

**Tasks covered:** T02, T05–T20 &nbsp;|&nbsp; **GPU:** T4 (Colab) &nbsp;|&nbsp; **Expected GPU time:** ~2–3 h (resumable)

**Everything lives on Google Drive** under `MyDrive/SahayakLLM/` (data, splits, manifest, prompts, predictions, figures, results log, even the Hugging Face model/dataset cache). Nothing is written anywhere else. The base model, the Bitext dataset and the metric models (`roberta-large` for BERTScore) are downloaded straight from Hugging Face into the Drive cache on first use.

**Order of work in this notebook (note: prompt comparison runs *before* training, as planned):**

1. Setup: Drive, deps, GPU check, helper modules, 4-bit model load (T02)
2. Dataset: load, EDA, token lengths, flags, placeholders, audit (T05–T10)
3. Clean + split + manifest + eval-mini (T11–T15)
4. Metrics harness (T17)
5. Compare 3 system prompts on eval-mini → **freeze the best** (T20)
6. Build SFT files (T16) and few-shot exemplars with the frozen prompt
7. **B0** zero-shot and **B1** few-shot on eval-mini + test (T18, T19), results log

**Colab sessions disconnect.** Every generation step is resumable (finished rows are cached on Drive), and every data step is skipped/overwritten safely. If the session dies, reconnect and *Run all* again.

> Dataset attribution (required): Bitext Customer Support LLM Chatbot Training Dataset, Bitext Innovations, CDLA-Sharing-1.0. KharidiKart is a fictional company.

## 0. Configuration (edit this cell only)

In [ ]:
# ===================== EDIT THESE =====================
TEAM_NUMBER = 10         # Faker seed for the DB in Phase B
HELD_OUT_INTENTS = ["switch_account", "review", "check_invoice"]  # simple, distinct intents; each keeps a sibling intent in its category
SEED = 42                # global seed (T04)

DRIVE_ROOT = "/content/drive/MyDrive/SahayakLLM"
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"        # fallback: "Qwen/Qwen2.5-0.5B-Instruct"
DATASET_ID = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"

TEST_LIMIT = None            # None = full test split. Set e.g. 200 for a quick dry run first.
RUN_HELDOUT_BASELINES = False  # True also scores B0/B1 on the held-out-intent set (~+1 h)
EVAL_MINI_N = 300
PPL_N = 300                  # rows used for the perplexity estimate
LATENCY_N = 10               # single-sample latency probes per baseline
FORCE_VARIANT = None         # None = pick best automatically; or "V1" / "V2" / "V3"
GEN_BS_ZERO_SHOT = 32        # generation batch sizes (auto-halve on out-of-memory)
GEN_BS_FEW_SHOT = 16
GEN_BS_ROUTE = 48
# ======================================================

# Starter system prompt from the brief (Section 10) = prompt variant V1
STARTER_SYSTEM = (
    "You are Sahayak, the customer-support assistant of KharidiKart.\n"
    "Be polite, concise and specific. Never invent order details, dates, amounts\n"
    "or policy numbers. If a fact is needed that you do not have, write the\n"
    "placeholder in double braces, e.g. {{Order Number}}."
)

# The 27 intents listed in the brief (Task T06 confirms them against the downloaded data)
EXPECTED_INTENTS = [
    "cancel_order", "change_order", "change_shipping_address", "check_cancellation_fee", "check_invoice",
    "check_payment_methods", "check_refund_policy", "complaint", "contact_customer_service", "contact_human_agent",
    "create_account", "delete_account", "delivery_options", "delivery_period", "edit_account", "get_invoice",
    "get_refund", "newsletter_subscription", "payment_issue", "place_order", "recover_password",
    "registration_problems", "review", "set_up_shipping_address", "switch_account", "track_order", "track_refund",
]
assert len(HELD_OUT_INTENTS) == 3 and set(HELD_OUT_INTENTS) <= set(EXPECTED_INTENTS), "check HELD_OUT_INTENTS"

## 1. Setup — Drive, dependencies, GPU (T02)

In [ ]:
# Mount Google Drive and create the project folder tree. ALL outputs go here.
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path
ROOT = Path(DRIVE_ROOT)
D = {k: ROOT / k for k in ["config", "data/splits", "data/analysis", "prompts", "preds", "results", "figures",
                            "reports", "docs", "src/data", "eval", "adapters", "hf_cache"]}
for p in D.values():
    p.mkdir(parents=True, exist_ok=True)

# Model + dataset + metric-model downloads are cached on Drive too (persist across sessions).
# Must be set BEFORE transformers / datasets are imported.
os.environ["HF_HOME"] = str(D["hf_cache"])
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Project root:", ROOT)

In [ ]:
# If pip upgrades transformers and the import below complains, use Runtime > Restart session and Run all again.
!pip install -q -U peft trl bitsandbytes accelerate rouge_score bert_score evaluate pyyaml

In [ ]:
import sys, json, re, math, time, hashlib, random, shutil, datetime, warnings
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=FutureWarning)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

!nvidia-smi
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME, "| bf16 supported:", torch.cuda.is_bf16_supported(), "(T4: no -> we use float16)")

### Helper modules (written to Drive so Notebooks 2–6 can import them)

| File | Purpose |
|---|---|
| `src/data/clean.py` | deterministic cleaning (T11) |
| `src/data/split.py` | held-out intents, stratified split, eval-mini, SHA-256 manifest + `verify_manifest()` (T13–T15) |
| `eval/metrics.py` | the evaluation harness (T17) |
| `eval/generation.py` | 4-bit model loader, chat-template prompt builders, resumable greedy generation, latency probe |

In [ ]:
def write_module(rel_path, content):
    p = ROOT / rel_path
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding="utf-8")
    print("wrote", p.relative_to(ROOT))

In [ ]:
write_module('src/data/clean.py', r'''
# src/data/clean.py -- deterministic cleaning for the Bitext rows (T11)
import re
import pandas as pd

_QUOTES = {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"', "\u00a0": " ", "\u200b": ""}


def clean_text(s):
    if s is None or (isinstance(s, float) and s != s):
        return ""
    s = str(s)
    for k, v in _QUOTES.items():
        s = s.replace(k, v)
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)       # collapse runs of spaces/tabs, keep newlines (numbered lists)
    s = re.sub(r" *\n *", "\n", s)      # no spaces around newlines
    s = re.sub(r"\n{3,}", "\n\n", s)    # at most one blank line
    return s.strip()


def clean_dataframe(df):
    # expects columns: row_id, flags, instruction, category, intent, response
    log = {"rows_in": int(len(df))}
    df = df.copy()
    for col in ["instruction", "response"]:
        df[col] = df[col].map(clean_text)
    for col in ["flags", "intent", "category"]:
        df[col] = df[col].astype(str).str.strip()
    empty = (df["instruction"] == "") | (df["response"] == "")
    log["dropped_empty"] = int(empty.sum())
    df = df[~empty]
    before = len(df)
    df = df.drop_duplicates(subset=["instruction", "response", "intent"], keep="first")
    log["dropped_exact_duplicate_pairs"] = int(before - len(df))
    df = df.reset_index(drop=True)
    log["rows_out"] = int(len(df))
    return df, log
''')

In [ ]:
write_module('src/data/split.py', r'''
# src/data/split.py -- held-out intents, stratified 80/10/10 split, eval-mini, hashed manifest (T13-T15)
import json
import hashlib
import os
import pandas as pd
from sklearn.model_selection import train_test_split

SPLIT_FILES = ["train", "val", "test", "held_out_test", "eval_mini"]


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def make_splits(df, held_out_intents, seed=42):
    held = df[df["intent"].isin(held_out_intents)].copy()
    rest = df[~df["intent"].isin(held_out_intents)].copy()
    train, tmp = train_test_split(rest, test_size=0.2, stratify=rest["intent"], random_state=seed)
    val, test = train_test_split(tmp, test_size=0.5, stratify=tmp["intent"], random_state=seed)
    return {"train": train, "val": val, "test": test, "held_out_test": held}


def make_eval_mini(val, n=300, seed=42):
    mini, _ = train_test_split(val, train_size=n, stratify=val["intent"], random_state=seed)
    return mini


def write_jsonl(df, path):
    df = df.sort_values("row_id")
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        for rec in df.to_dict("records"):
            f.write(json.dumps(rec, ensure_ascii=False, default=int) + "\n")


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return pd.DataFrame([json.loads(line) for line in f if line.strip()])


def manifest_hash(seed, held_out_intents, file_hashes):
    payload = json.dumps({"seed": seed, "held_out_intents": sorted(held_out_intents),
                          "files": file_hashes}, sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def build_manifest(split_dir, splits, held_out_intents, seed, extra=None):
    files, hashes = {}, {}
    for name in SPLIT_FILES:
        path = os.path.join(split_dir, name + ".jsonl")
        write_jsonl(splits[name], path)
        sha = sha256_file(path)
        hashes[name] = sha
        files[name] = {"file": name + ".jsonl", "rows": int(len(splits[name])), "sha256": sha,
                       "per_intent": {k: int(v) for k, v in splits[name]["intent"].value_counts().sort_index().items()}}
    m = {"seed": seed, "held_out_intents": sorted(held_out_intents), "files": files,
         "manifest_hash": manifest_hash(seed, held_out_intents, hashes)}
    if extra:
        m.update(extra)
    with open(os.path.join(split_dir, "manifest.json"), "w") as f:
        json.dump(m, f, indent=2)
    return m


def verify_manifest(split_dir):
    # Notebooks 2 and 3 call this instead of recomputing the data. Raises if any file changed.
    with open(os.path.join(split_dir, "manifest.json")) as f:
        m = json.load(f)
    hashes = {}
    for name, info in m["files"].items():
        actual = sha256_file(os.path.join(split_dir, info["file"]))
        if actual != info["sha256"]:
            raise RuntimeError("Split file changed: " + info["file"])
        hashes[name] = actual
    if manifest_hash(m["seed"], m["held_out_intents"], hashes) != m["manifest_hash"]:
        raise RuntimeError("Manifest hash mismatch")
    return m
''')

In [ ]:
write_module('eval/metrics.py', r'''
# eval/metrics.py -- evaluation harness (T17): intent F1, ROUGE-L, BERTScore, perplexity, leakage, latency
import re
import json
import numpy as np

PLACEHOLDER_RE = re.compile(r"\{\{.*?\}\}")
INVALID = "__INVALID__"


def extract_placeholders(text):
    return PLACEHOLDER_RE.findall(text or "")


def has_placeholder(text):
    return bool(PLACEHOLDER_RE.search(text or ""))


def placeholder_leakage(responses):
    responses = list(responses)
    if not responses:
        return 0.0
    return 100.0 * float(np.mean([has_placeholder(r) for r in responses]))


def parse_route_json(text, valid_intents=None):
    # returns (intent, category); (None, None) if nothing usable
    if text is None:
        return None, None
    m = re.search(r"\{.*?\}", text, re.S)
    if m:
        try:
            obj = json.loads(m.group(0))
            intent = str(obj.get("intent", "")).strip()
            cat = str(obj.get("category", "")).strip()
            if intent:
                return intent, (cat or None)
        except Exception:
            pass
    m = re.search(r"intent\W{0,4}\s*[:=]\s*\W{0,2}([a-z_]+)", text)
    if m:
        return m.group(1), None
    if valid_intents:
        low = text.lower()
        hits = [i for i in valid_intents if i in low]
        if len(hits) == 1:
            return hits[0], None
    return None, None


def intent_metrics(gold, pred, labels=None):
    from sklearn.metrics import accuracy_score, f1_score
    gold = list(gold)
    pred = [p if p is not None else INVALID for p in pred]
    labels = sorted(set(labels) if labels is not None else set(gold))
    return {
        "intent_acc": float(accuracy_score(gold, pred)),
        "intent_macro_f1": float(f1_score(gold, pred, labels=labels, average="macro", zero_division=0)),
        "invalid_rate": float(np.mean([p == INVALID for p in pred])),
    }


def rouge_l(preds, refs):
    from rouge_score import rouge_scorer
    sc = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    f = [sc.score(r, p)["rougeL"].fmeasure for p, r in zip(preds, refs)]
    return float(np.mean(f)), f


_BERT = {}


def bertscore_f1(preds, refs, batch_size=32, model_type="roberta-large", device="cuda"):
    from bert_score import BERTScorer
    key = (model_type, device)
    if key not in _BERT:
        _BERT[key] = BERTScorer(model_type=model_type, lang="en", device=device)
    preds = [p if p.strip() else "." for p in preds]
    _, _, F = _BERT[key].score(preds, list(refs), batch_size=batch_size)
    return float(F.mean()), F.tolist()


def response_perplexity(model, tok, items, batch_size=4, max_len=1024):
    # items: list of (messages_without_assistant, reference_response). Loss on response tokens only.
    import torch
    model.eval()
    tot_nll, tot_tok = 0.0, 0
    pad = tok.pad_token_id
    for i in range(0, len(items), batch_size):
        seqs, labs = [], []
        for msgs, resp in items[i:i + batch_size]:
            ptxt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            p_ids = tok(ptxt, add_special_tokens=False)["input_ids"]
            r_ids = tok(resp + tok.eos_token, add_special_tokens=False)["input_ids"]
            ids = (p_ids + r_ids)[:max_len]
            lab = ([-100] * len(p_ids) + r_ids)[:max_len]
            seqs.append(ids)
            labs.append(lab)
        L = max(len(s) for s in seqs)
        ids_t = torch.tensor([s + [pad] * (L - len(s)) for s in seqs], device=model.device)
        att_t = torch.tensor([[1] * len(s) + [0] * (L - len(s)) for s in seqs], device=model.device)
        lab_t = torch.tensor([l + [-100] * (L - len(l)) for l in labs], device=model.device)
        with torch.no_grad():
            logits = model(input_ids=ids_t, attention_mask=att_t).logits[:, :-1, :].float()
        tgt = lab_t[:, 1:]
        loss = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1),
                                                 ignore_index=-100, reduction="sum")
        tot_nll += float(loss.item())
        tot_tok += int((tgt != -100).sum().item())
        del logits
    return float(np.exp(tot_nll / max(tot_tok, 1)))


def latency_stats(seconds):
    a = np.asarray(list(seconds), dtype=float)
    return {"p50_s": float(np.percentile(a, 50)), "p95_s": float(np.percentile(a, 95))}
''')

In [ ]:
write_module('eval/generation.py', r'''
# eval/generation.py -- model loading, chat-template prompt builders, resumable greedy batch generation
import json
import os
import time
import numpy as np
import torch

ROUTER_TEMPLATE = (
    'You are the intent router of the KharidiKart customer-support assistant. '
    'Classify the customer message into exactly one intent from this list: {intents}. '
    'The category must be one of: {categories}. '
    'Reply with only a JSON object like {{"intent": "<intent>", "category": "<CATEGORY>"}} and nothing else.'
)


def make_router_system(intents, categories):
    return ROUTER_TEMPLATE.format(intents=", ".join(sorted(intents)), categories=", ".join(sorted(categories)))


def route_target(intent, category):
    return json.dumps({"intent": intent, "category": category})


def build_response_messages(system, instruction, exemplars=None):
    msgs = [{"role": "system", "content": system}]
    for ex in exemplars or []:
        msgs.append({"role": "user", "content": ex["instruction"]})
        msgs.append({"role": "assistant", "content": ex["response"]})
    msgs.append({"role": "user", "content": instruction})
    return msgs


def build_route_messages(router_system, instruction, exemplars=None):
    msgs = [{"role": "system", "content": router_system}]
    for ex in exemplars or []:
        msgs.append({"role": "user", "content": ex["instruction"]})
        msgs.append({"role": "assistant", "content": route_target(ex["intent"], ex["category"])})
    msgs.append({"role": "user", "content": instruction})
    return msgs


def render(tok, messages):
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def load_model_4bit(model_id, adapter_path=None, cache_dir=None):
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    tok = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kw = dict(quantization_config=bnb, device_map={"": 0}, cache_dir=cache_dir)
    try:
        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, **kw)
    except (TypeError, ValueError):
        model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, **kw)
    if adapter_path:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter_path)
    model.eval()
    return tok, model


def _run_batch(model, tok, texts, max_new_tokens):
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, temperature=None,
                             top_p=None, top_k=None, repetition_penalty=1.0, pad_token_id=tok.pad_token_id)
    dt = time.time() - t0
    plen = enc["input_ids"].shape[1]
    return out[:, plen:].cpu(), dt, plen


def _gen_texts(model, tok, texts, max_new_tokens):
    try:
        gen, dt, plen = _run_batch(model, tok, texts, max_new_tokens)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(texts) == 1:
            raise
        h = len(texts) // 2
        return _gen_texts(model, tok, texts[:h], max_new_tokens) + _gen_texts(model, tok, texts[h:], max_new_tokens)
    res = []
    for j in range(len(texts)):
        toks = gen[j]
        n_new = int((toks != tok.pad_token_id).sum())
        res.append((tok.decode(toks, skip_special_tokens=True).strip(), n_new, dt / len(texts), plen))
    return res


def generate_chat(model, tok, messages_list, ids, max_new_tokens, batch_size, out_path):
    # Greedy, chat-template based, resumable: finished ids are read back from out_path and skipped.
    from tqdm.auto import tqdm
    done = {}
    if os.path.exists(out_path):
        with open(out_path, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    r = json.loads(line)
                    done[r["id"]] = r
    todo = [i for i in range(len(messages_list)) if ids[i] not in done]
    texts = {i: render(tok, messages_list[i]) for i in todo}
    order = sorted(todo, key=lambda i: len(texts[i]))
    tok.padding_side = "left"
    with open(out_path, "a", encoding="utf-8") as f:
        for b in tqdm(range(0, len(order), batch_size), desc=os.path.basename(out_path)):
            idx = order[b:b + batch_size]
            res = _gen_texts(model, tok, [texts[i] for i in idx], max_new_tokens)
            for i, (txt, n_new, share, plen) in zip(idx, res):
                rec = {"id": ids[i], "output": txt, "n_new_tokens": n_new,
                       "batch_share_s": round(share, 4), "prompt_tokens": plen}
                done[ids[i]] = rec
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            f.flush()
    return [done[ids[i]] for i in range(len(messages_list))]


def latency_benchmark(model, tok, messages_list, max_new_tokens, n=10, warmup=1):
    # batch size 1; returns per-sample seconds, tokens/s and peak GPU memory (GB)
    for m in messages_list[:warmup]:
        _gen_texts(model, tok, [render(tok, m)], 16)
    torch.cuda.reset_peak_memory_stats()
    secs, toks = [], 0
    for m in messages_list[warmup:warmup + n]:
        txt, n_new, dt, _ = _gen_texts(model, tok, [render(tok, m)], max_new_tokens)[0]
        secs.append(dt)
        toks += n_new
    return {"seconds": secs, "tokens_per_s": float(toks / max(sum(secs), 1e-9)),
            "peak_gpu_gb": float(torch.cuda.max_memory_allocated() / 1e9)}
''')

In [ ]:
sys.path[:0] = [str(ROOT / "src/data"), str(ROOT / "eval")]
import importlib, clean, split, metrics, generation
for m in (clean, split, metrics, generation):
    importlib.reload(m)
from clean import clean_dataframe
from split import (make_splits, make_eval_mini, build_manifest, verify_manifest, read_jsonl, write_jsonl,
                   sha256_file)
from metrics import (intent_metrics, parse_route_json, rouge_l, bertscore_f1, response_perplexity,
                     placeholder_leakage, extract_placeholders, latency_stats, has_placeholder)
from generation import (load_model_4bit, make_router_system, build_response_messages, build_route_messages,
                        route_target, generate_chat, latency_benchmark, render)

### T02 — Load the model in 4-bit (NF4)
Weights are fetched directly from Hugging Face into the Drive cache (first run ≈ 3 GB).

In [ ]:
tok, model = load_model_4bit(BASE_MODEL)
print(f"Model loaded in 4-bit on {GPU_NAME} | GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Pin the environment (requirements.txt) + record GPU info
from importlib.metadata import version, PackageNotFoundError
PKGS = ["transformers", "peft", "trl", "bitsandbytes", "accelerate", "datasets", "evaluate", "rouge_score",
        "bert_score", "scikit-learn", "pandas", "numpy", "matplotlib", "pyyaml", "huggingface_hub",
        "tokenizers", "safetensors"]
lines = [f"# python {sys.version.split()[0]}", f"# torch=={torch.__version__}  (preinstalled on Colab; do not reinstall)"]
vers = {}
for p in PKGS:
    try:
        vers[p] = version(p); lines.append(f"{p}=={vers[p]}")
    except PackageNotFoundError:
        pass
(ROOT / "requirements.txt").write_text("\n".join(lines) + "\n")
(D["config"] / "env_info.json").write_text(json.dumps(
    {"gpu": GPU_NAME, "torch": torch.__version__, "python": sys.version.split()[0], "packages": vers,
     "date": str(datetime.date.today())}, indent=2))
print("\n".join(lines))

### T03 / T04 stubs — team config and experiment-log template
Minimal versions so this notebook can log results; S4 can refine them later.

In [ ]:
import yaml
(D["config"] / "team.yaml").write_text(yaml.safe_dump(
    {"team_number": TEAM_NUMBER, "held_out_intents": HELD_OUT_INTENTS, "global_seed": SEED,
     "base_model": BASE_MODEL, "dataset": DATASET_ID}, sort_keys=False))

RESULT_COLS = ["run_id", "config", "split", "data_manifest_hash", "seed", "prompt_hash", "n",
               "intent_acc", "intent_macro_f1", "invalid_json_rate", "rougeL", "bertscore_f1", "perplexity",
               "leak_pct", "avg_resp_tokens", "trunc_pct", "p50_s", "p95_s", "tokens_per_s", "peak_gpu_gb",
               "tsr", "tool_acc", "halluc_pct", "date", "notes"]
pd.DataFrame(columns=RESULT_COLS).to_csv(ROOT / "eval/results_template.csv", index=False)
RESULTS_PATH = D["results"] / "results_log.csv"

def log_row(row):
    df = pd.read_csv(RESULTS_PATH) if RESULTS_PATH.exists() else pd.DataFrame(columns=RESULT_COLS)
    df = df[df["run_id"] != row["run_id"]]
    row = {c: row.get(c, np.nan) for c in RESULT_COLS}
    row["date"] = str(datetime.datetime.now())[:19]
    row["seed"] = SEED
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(RESULTS_PATH, index=False)
print(open(D["config"] / "team.yaml").read())

## 2. Dataset understanding (T05–T10)
### T05 — Load Bitext straight from Hugging Face

In [ ]:
from datasets import load_dataset
raw = load_dataset(DATASET_ID, split="train")     # cached in Drive via HF_HOME
df_raw = raw.to_pandas()
need = {"flags", "instruction", "category", "intent", "response"}
assert need <= set(df_raw.columns), f"unexpected columns: {df_raw.columns.tolist()}"
df_raw.insert(0, "row_id", range(len(df_raw)))     # stable id for every original row
print("rows:", len(df_raw), "(brief says 26,872) | columns:", df_raw.columns.tolist())
print("\ndtypes:\n", df_raw.dtypes)
print("\nnull counts:\n", df_raw.isna().sum())
pd.set_option("display.max_colwidth", 90)
df_raw.sample(10, random_state=SEED)[["row_id", "flags", "instruction", "category", "intent", "response"]]

### T06 — Intent and category distributions; confirm the intent list

In [ ]:
intent_counts = df_raw["intent"].value_counts().sort_index()
cat_counts = df_raw["category"].value_counts().sort_index()
found, expected = set(df_raw["intent"]), set(EXPECTED_INTENTS)
print(f"{len(found)} intents, {len(cat_counts)} categories")
print("In brief but NOT in data:", sorted(expected - found) or "none")
print("In data but NOT in brief:", sorted(found - expected) or "none")
assert set(HELD_OUT_INTENTS) <= found, "held-out intents must exist in the data"

fig, ax = plt.subplots(1, 2, figsize=(15, 7), gridspec_kw={"width_ratios": [1.6, 1]})
intent_counts.sort_values().plot.barh(ax=ax[0], color="#3b6ea5"); ax[0].set_title("Rows per intent"); ax[0].set_xlabel("rows")
cat_counts.sort_values().plot.barh(ax=ax[1], color="#c0713b"); ax[1].set_title("Rows per category"); ax[1].set_xlabel("rows")
plt.tight_layout(); plt.savefig(D["figures"] / "T06_intent_category_distribution.png", dpi=150); plt.show()

by_cat = df_raw.groupby("category")["intent"].unique().apply(sorted)
for c, its in by_cat.items():
    print(f"{c:13s} {len(its):2d} intents: {', '.join(its)}")
intent_counts.rename("rows").to_csv(D["data/analysis"] / "intent_counts.csv")

### T07 — Token lengths with the Qwen tokenizer → `max_seq_length`
A chat-formatted training sample = system + instruction + response + template tokens. We choose the smallest multiple of 32 covering ≥ 98 % of samples. (Re-checked in T16 against the *frozen* system prompt.)

In [ ]:
instr_len = np.array([len(x) for x in tok(df_raw["instruction"].tolist(), add_special_tokens=False)["input_ids"]])
resp_len = np.array([len(x) for x in tok(df_raw["response"].tolist(), add_special_tokens=False)["input_ids"]])
overhead = len(tok(tok.apply_chat_template(
    [{"role": "system", "content": STARTER_SYSTEM}, {"role": "user", "content": ""}, {"role": "assistant", "content": ""}],
    tokenize=False), add_special_tokens=False)["input_ids"])
full_len = instr_len + resp_len + overhead

def ceil32(x): return int(math.ceil(x / 32) * 32)
MAX_SEQ = ceil32(np.percentile(full_len, 98))
while (full_len <= MAX_SEQ).mean() < 0.98:
    MAX_SEQ += 32
MAX_NEW_RESP = min(320, ceil32(np.percentile(resp_len, 95)))   # generation cap used for ALL configs
MAX_NEW_ROUTE = 48
cover = (full_len <= MAX_SEQ).mean()

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for a, arr, name in zip(ax, [instr_len, resp_len, full_len], ["instruction tokens", "response tokens", "full chat sample tokens"]):
    a.hist(arr, bins=60, color="#3b6ea5"); a.set_title(name); a.axvline(np.percentile(arr, 98), color="r", ls="--", label="p98"); a.legend()
plt.tight_layout(); plt.savefig(D["figures"] / "T07_token_lengths.png", dpi=150); plt.show()

pct = {f"p{q}": float(np.percentile(full_len, q)) for q in (50, 90, 95, 98, 99, 100)}
print("full-sample length percentiles:", pct)
print(f"response tokens: mean {resp_len.mean():.0f}, p95 {np.percentile(resp_len,95):.0f}, max {resp_len.max()}")
print(f"\nmax_seq_length = {MAX_SEQ}  (covers {cover:.2%} of pairs) | generation cap max_new_tokens = {MAX_NEW_RESP}")
print(f"Justification: p98 of the chat-formatted length is {pct['p98']:.0f} tokens; {MAX_SEQ} is the next multiple of 32, "
      f"covering {cover:.2%} (>= 98%) of pairs while keeping T4 memory low.")
(D["config"] / "data_config.json").write_text(json.dumps(
    {"max_seq_length": MAX_SEQ, "max_new_tokens_resp": MAX_NEW_RESP, "max_new_tokens_route": MAX_NEW_ROUTE,
     "length_percentiles": pct}, indent=2))

### T08 — `flags` column: meaning and count of every tag
Tag meanings are from the Bitext dataset card (language-generation tags). The cell also downloads the card's README from Hugging Face and prints its tag lines so you can verify the table below against the source.

In [ ]:
FLAG_MEANING = {
    "B": "Basic syntactic structure", "C": "Coordinated syntactic structure", "I": "Interrogative structure",
    "N": "Negation (don't, can't...)", "M": "Morphological variation (plurals, tenses...)",
    "L": "Lexical variation (synonyms)", "E": "Expanded abbreviations (I'm -> I am)",
    "Q": "Colloquial variation", "P": "Politeness variation", "W": "Offensive language",
    "K": "Keyword language", "Z": "Noise (spelling / punctuation errors)",
    "S": "Syntactic structure (older card versions)", "D": "Indirect speech (older card versions)",
}
flag_chars = df_raw["flags"].fillna("").map(lambda s: sorted(set(s)))
rows = []
for ch in sorted(set(c for l in flag_chars for c in l)):
    n = int(flag_chars.map(lambda l: ch in l).sum())
    rows.append({"tag": ch, "meaning": FLAG_MEANING.get(ch, "NOT IN CARD TABLE - check card"), "rows": n,
                 "pct_rows": round(100 * n / len(df_raw), 2)})
flag_table = pd.DataFrame(rows).sort_values("rows", ascending=False).reset_index(drop=True)
flag_table.to_csv(D["data/analysis"] / "flags_table.csv", index=False)
display(flag_table)
print("Distinct flag strings:", df_raw["flags"].nunique(), "| top combos:\n", df_raw["flags"].value_counts().head(10).to_string())

flag_table.set_index("tag")["rows"].plot.bar(figsize=(9, 3.5), color="#3b6ea5", title="Rows carrying each tag")
plt.tight_layout(); plt.savefig(D["figures"] / "T08_flags.png", dpi=150); plt.show()

# Print the tag lines from the dataset card itself (source of truth)
try:
    from huggingface_hub import hf_hub_download
    card = open(hf_hub_download(DATASET_ID, "README.md", repo_type="dataset")).read()
    print("\n--- tag lines found in the dataset card ---")
    for ln in card.splitlines():
        if re.match(r"^\s*[-*]?\s*[A-Z]\s+-\s+\S", ln):
            print(ln.strip())
except Exception as e:
    print("Could not fetch the card:", e)

### T09 — Placeholders: extract `{{...}}`, frequency table, map each to a source
The mapping below is a **heuristic first proposal** (keyword rules). Open `data/analysis/placeholder_map.csv` and correct anything marked `UNMAPPED` or that looks wrong — this file drives the Action Agent later.

In [ ]:
ph_rows = []
for rid, intent, resp, ins in zip(df_raw["row_id"], df_raw["intent"], df_raw["response"], df_raw["instruction"]):
    for ph in set(extract_placeholders(resp)):
        ph_rows.append((ph.strip("{}").strip(), intent))
ph_df = pd.DataFrame(ph_rows, columns=["placeholder", "intent"])
freq = ph_df.groupby("placeholder").agg(count=("intent", "size"), n_intents=("intent", "nunique"),
        typical_intents=("intent", lambda s: ", ".join(s.value_counts().head(3).index))).sort_values("count", ascending=False)

# (substring rules, source_type, source_ref) - first match wins
RULES = [
    (["order number"], "DB", "orders.order_id"),
    (["refund amount"], "DB", "refunds.amount_inr"),
    (["refund id", "refund number"], "DB", "refunds.refund_id"),
    (["invoice number", "invoice id"], "DB", "invoices.invoice_id"),
    (["money amount", "amount", "price", "total"], "DB", "orders.amount_inr"),
    (["customer support phone", "phone number", "phone"], "CONFIG", "support_phone"),
    (["customer support email", "support email"], "CONFIG", "support_email"),
    (["email"], "DB", "customers.email"),
    (["company", "store", "website", "web site", "url", "link", "portal", "navigation", "channel", "chat",
      "app", "section", "tab", "button", "menu", "page", "settings", "profile", "option", "search", "form"],
     "CONFIG", "ui_navigation_text"),
    (["first name", "last name", "client name", "customer name", "name"], "DB", "customers.name"),
    (["shipping address", "delivery address", "address"], "DB", "orders.shipping_address"),
    (["payment method", "payment"], "DB", "orders.payment_mode"),
    (["delivery period", "time frame", "timeframe", "days"], "KB", "delivery_periods.md"),
    (["date", "eta"], "DB", "orders.eta / orders.placed_at"),
    (["hours", "working", "business"], "KB", "human_agent_hours.md"),
    (["fee", "charge"], "KB", "cancellation_and_fees.md"),
    (["policy", "refund"], "KB", "refund_policy.md"),
    (["account"], "DB", "customers.*"),
]
def map_placeholder(name):
    low = name.lower()
    for keys, typ, ref in RULES:
        if any(k in low for k in keys):
            return typ, ref
    return "UNMAPPED", "needs manual review"
freq[["source_type", "source_ref"]] = [map_placeholder(n) for n in freq.index]
freq.reset_index().to_csv(D["data/analysis"] / "placeholder_freq_full.csv", index=False)
top30 = freq.head(30).reset_index()
top30.to_csv(D["data/analysis"] / "placeholder_map.csv", index=False)
print(f"{len(freq)} distinct placeholders | responses containing >=1: {df_raw['response'].map(has_placeholder).mean():.1%} "
      f"| instructions containing >=1: {df_raw['instruction'].map(has_placeholder).mean():.1%}")
display(top30)
print("Top-30 UNMAPPED:", int((top30['source_type'] == 'UNMAPPED').sum()))
TOP_PLACEHOLDERS = freq.head(12).index.tolist()

### T10 — Data audit (auto-generated report → `reports/data_audit.md`)

In [ ]:
def norm(s): return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9{} ]+", " ", s.lower())).strip()

n = len(df_raw)
dup_pairs = int(df_raw.duplicated(subset=["instruction", "response", "intent"]).sum())
dup_instr = int(df_raw.duplicated(subset=["instruction"]).sum())
g = df_raw.groupby("instruction")["response"].nunique()
instr_multi_resp = int((g > 1).sum())
df_raw["_ni"] = df_raw["instruction"].map(norm)
near_dup_extra = int(df_raw.duplicated(subset=["_ni"]).sum()) - dup_instr
df_raw.drop(columns="_ni", inplace=True)
cnt = df_raw["intent"].value_counts()
wlen = df_raw["response"].str.split().str.len()
short = df_raw[wlen < 20]
contact_re = re.compile(r"https?://|www\.|[\w.]+@[\w.]+\.\w+|\+?\d[\d \-]{8,}\d")
contact_rows = int(df_raw["response"].str.contains(contact_re).sum())
noise_rows = float(df_raw["flags"].fillna("").str.contains("Z").mean())

report = f"""# Data audit - Bitext Customer Support dataset (auto-generated {datetime.date.today()})

Source: `{DATASET_ID}` (CDLA-Sharing-1.0, Bitext Innovations). Hybrid synthetic data: no real customer data.

## Size and structure
- Rows: **{n:,}**, columns: {', '.join(df_raw.columns[1:])}. Null cells: {int(df_raw.isna().sum().sum())}.
- Intents: **{cnt.size}**, categories: **{df_raw['category'].nunique()}**.

## Duplicates
- Exact duplicate (instruction, response, intent) pairs: **{dup_pairs}**
- Rows whose instruction text already appears earlier: **{dup_instr}** (of which {instr_multi_resp} instruction strings have >1 distinct response)
- Extra near-duplicates after lower-casing/stripping punctuation: **{near_dup_extra}**

## Class balance
- Rows per intent: min **{cnt.min()}**, median **{int(cnt.median())}**, max **{cnt.max()}** (max/min = {cnt.max()/cnt.min():.2f}) -> essentially balanced.

## Responses
- Words per response: min {wlen.min()}, median {int(wlen.median())}, p95 {int(wlen.quantile(.95))}, max {wlen.max()}.
- Responses with fewer than 20 words: {len(short)}.
- Responses containing a `{{{{placeholder}}}}`: **{df_raw['response'].map(has_placeholder).mean():.1%}**; distinct placeholder names: {len(freq)}.
- Responses containing a URL / e-mail / phone-like string: {contact_rows}.
- Instructions flagged with noise (typos, tag Z): {noise_rows:.1%}.

## Risks and how the project handles them
1. **Synthetic, templated language** - many responses share boilerplate openings, so ROUGE-L/BERTScore can look good without real understanding. Mitigation: report them next to intent F1, TSR and human evaluation.
2. **Near-paraphrase leakage** - a random split would put paraphrases of one query in train and test. Mitigation: held-out intents + stratified split + cross-split overlap check in the manifest step.
3. **Placeholders instead of facts** - the model learns where facts go but not the facts. Mitigation: keep slots in targets (T12), fill them with tools, measure leakage.
4. **Generic policies** - Bitext responses contain no KharidiKart-specific numbers, so hallucination of policy numbers cannot be judged from this data alone. Mitigation: our own knowledge base with concrete numbers.
5. **Held-out intents remove whole categories** - a held-out intent that is the only member of its category removes that category from training.
6. **Small number of distinct intents (27) and clean label noise** - intent F1 on this data is likely to be high for a fine-tuned model; do not over-interpret it.
"""
(D["reports"] / "data_audit.md").write_text(report, encoding="utf-8")
print(report)
print("Shortest responses:"); display(short.head(5)[["intent", "response"]])

## 3. Clean, split, manifest (T11–T15)
### T11 — Clean text and drop exact duplicate pairs

In [ ]:
df, clean_log = clean_dataframe(df_raw)
print("cleaning log:", clean_log)
# deterministic re-run check
df2, _ = clean_dataframe(df_raw)
assert df.equals(df2), "cleaning must be deterministic"
(D["data/analysis"] / "cleaning_log.json").write_text(json.dumps(clean_log, indent=2))

### T12 — Placeholder strategy (decision note)
**Decision:** keep `{{slots}}` in the fine-tuning targets so the model learns *where* facts belong; the Action Agent fills them from tool output and the Verifier blocks any leftover slot. This is written to `docs/decisions.md` for S1 and S3 to sign off.

In [ ]:
(D["docs"] / "decisions.md").write_text(f"""# Design decisions

## D1 - Placeholder strategy (T12)
Keep {{{{slots}}}} in training targets. The model learns where facts go; agents fill them from verified tool output;
the Verifier rejects any final answer that still contains a slot. Placeholder-leakage rate (C2 vs C4) quantifies the gap.
Sign-off: S1 [ ]  S3 [ ]

## D2 - Router prompt lists all 27 intents
The routing system prompt lists all 27 intent names (including the 3 held-out ones) so that held-out generalisation
(RQ5) measures whether the fine-tuned model can route to a label it never saw as a training target.

## D3 - Decoding
Greedy decoding (do_sample=False, repetition_penalty=1.0 set explicitly; Qwen's default 1.05 is NOT used) for every reported number.
""", encoding="utf-8")
print("saved docs/decisions.md")

### T13 + T14 + T15 — Held-out intents, stratified 80/10/10 split, eval-mini, hashed manifest

In [ ]:
SPLIT_DIR = D["data/splits"]
splits = make_splits(df, HELD_OUT_INTENTS, SEED)
splits["eval_mini"] = make_eval_mini(splits["val"], EVAL_MINI_N, SEED)

# sanity checks required by the brief
assert not (set(splits["train"].intent) | set(splits["val"].intent) | set(splits["test"].intent)) & set(HELD_OUT_INTENTS)
assert set(splits["held_out_test"].intent) == set(HELD_OUT_INTENTS)
n_int = df[~df.intent.isin(HELD_OUT_INTENTS)].intent.nunique()
for k in ("train", "val", "test", "eval_mini"):
    assert splits[k].intent.nunique() == n_int, f"{k} misses an intent"

# cross-split overlap of identical instruction text (reported, not removed - brief only requires exact-pair dedup)
tr_i = set(splits["train"].instruction)
overlap = {k: int(splits[k].instruction.isin(tr_i).sum()) for k in ("val", "test")}
print("instruction strings in val/test that also appear in train:", overlap)

manifest = build_manifest(str(SPLIT_DIR), splits, HELD_OUT_INTENTS, SEED,
                          extra={"team_number": TEAM_NUMBER, "source_dataset": DATASET_ID, "cleaning_log": clean_log,
                                 "cross_split_instruction_overlap": overlap,
                                 "protocol": "held-out intents; stratified 80/10/10 on remaining intents; eval_mini = 300 stratified rows of val"})
MANIFEST_HASH = manifest["manifest_hash"]
print({k: v["rows"] for k, v in manifest["files"].items()})
print("MANIFEST HASH:", MANIFEST_HASH)

In [ ]:
# From here on we use ONLY the files on Drive, exactly as Notebooks 2 and 3 will (verify hash, then load).
m = verify_manifest(str(SPLIT_DIR))
assert m["manifest_hash"] == MANIFEST_HASH
tr, va, te, ho, mini = [read_jsonl(SPLIT_DIR / f"{n}.jsonl") for n in ("train", "val", "test", "held_out_test", "eval_mini")]
print("manifest verified:", MANIFEST_HASH[:16], "...")
print(f"train {len(tr)} | val {len(va)} | test {len(te)} | held-out {len(ho)} | eval-mini {len(mini)}")
ALL_INTENTS = sorted(df.intent.unique()); ALL_CATEGORIES = sorted(df.category.unique())

## 4. Evaluation harness (T17)
`eval/metrics.py` (written to Drive above) provides intent accuracy / macro-F1, ROUGE-L, BERTScore-F1, perplexity, placeholder leakage and latency percentiles. It is used by every later notebook.

## 5. Prompt variants → freeze the best (T20)
Three system prompts, zero-shot, on **eval-mini only** (the test set is not touched):
* **V1** – the starter prompt from the brief
* **V2** – structured rules (acknowledge → steps → no invention → placeholders → close)
* **V3** – V2 + the dataset's most frequent placeholder names, so the model reuses the slot vocabulary the agents will fill

Selection rule: highest BERTScore-F1 on eval-mini (ROUGE-L breaks ties). Set `FORCE_VARIANT` in the config to override.

In [ ]:
V2 = (
    "You are Sahayak, the customer-support assistant of KharidiKart, an online retailer in India.\n"
    "Follow these rules in every reply:\n"
    "1. Open with one short sentence that acknowledges the customer's request.\n"
    "2. Give the answer, or numbered steps when a procedure is involved. Be polite, concise and specific.\n"
    "3. Never invent order details, dates, amounts, refund ids or policy numbers.\n"
    "4. If a specific fact is needed that you do not have, write it as a placeholder in double braces, e.g. {{Order Number}}.\n"
    "5. Close by offering further help."
)
V3 = V2.replace(
    "e.g. {{Order Number}}.",
    "using one of these exact names: " + ", ".join("{{" + p + "}}" for p in TOP_PLACEHOLDERS) + ".")
VARIANTS = {"V1": STARTER_SYSTEM, "V2": V2, "V3": V3}
ROUTER_SYSTEM = make_router_system(ALL_INTENTS, ALL_CATEGORIES)
for k, v in VARIANTS.items():
    print(f"----- {k} -----\n{v}\n")

In [ ]:
def score_responses(preds, refs):
    rl, _ = rouge_l(preds, refs)
    bf, _ = bertscore_f1(preds, refs)
    return {"rougeL": rl, "bertscore_f1": bf, "leak_pct": placeholder_leakage(preds),
            "avg_words": float(np.mean([len(p.split()) for p in preds]))}

rows = []
ids_mini = mini["row_id"].tolist()
for name, system in VARIANTS.items():
    msgs = [build_response_messages(system, ins) for ins in mini["instruction"]]
    recs = generate_chat(model, tok, msgs, ids_mini, MAX_NEW_RESP, GEN_BS_ZERO_SHOT,
                         str(D["preds"] / f"promptcmp_{name}_evalmini_response.jsonl"))
    outs = [r["output"] for r in recs]
    s = score_responses(outs, mini["response"].tolist())
    s.update({"variant": name, "trunc_pct": 100 * float(np.mean([r["n_new_tokens"] >= MAX_NEW_RESP for r in recs])),
              "prompt_hash": hashlib.sha256(system.encode()).hexdigest()[:12]})
    rows.append(s)
    print(name, {k: round(v, 4) if isinstance(v, float) else v for k, v in s.items()})

cmp = pd.DataFrame(rows).set_index("variant")
cmp.to_csv(D["results"] / "prompt_comparison.csv")
display(cmp)

In [ ]:
best = FORCE_VARIANT or cmp.sort_values(["bertscore_f1", "rougeL"], ascending=False).index[0]
SYSTEM_PROMPT = VARIANTS[best]
PROMPT_HASH = hashlib.sha256(SYSTEM_PROMPT.encode()).hexdigest()
(D["prompts"] / "system.txt").write_text(SYSTEM_PROMPT, encoding="utf-8")
(D["prompts"] / "router_system.txt").write_text(ROUTER_SYSTEM, encoding="utf-8")
(D["prompts"] / "variants.json").write_text(json.dumps(VARIANTS, indent=2, ensure_ascii=False), encoding="utf-8")
(D["prompts"] / "frozen.json").write_text(json.dumps(
    {"chosen_variant": best, "selection_rule": "max BERTScore-F1 on eval-mini (ROUGE-L tie-break)" if not FORCE_VARIANT else "forced",
     "system_sha256": PROMPT_HASH, "router_sha256": hashlib.sha256(ROUTER_SYSTEM.encode()).hexdigest(),
     "data_manifest_hash": MANIFEST_HASH, "scores": cmp.round(4).to_dict("index")}, indent=2))
print(f"FROZEN PROMPT: {best}  (sha256 {PROMPT_HASH[:12]})  ->  prompts/system.txt")
print(SYSTEM_PROMPT)

## 6. SFT data with the frozen prompt (T16) and few-shot exemplars
* **Response records** (one per train row): system = frozen prompt, user = instruction, assistant = reference response.
* **Routing records**: a random 25 % of the train rows get a second record with the router system prompt and the target `{"intent": ..., "category": ...}` → routing is exactly **20 %** of all records.
* Chat formatting is applied with `tokenizer.apply_chat_template` at training time; three formatted samples are printed below.
* `max_seq_length` is re-checked against the frozen prompt.

In [ ]:
def sft_records(frame, route_frac=0.25, seed=SEED):
    recs = [{"id": int(r.row_id), "task": "response", "intent": r.intent,
             "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": r.instruction},
                          {"role": "assistant", "content": r.response}]} for r in frame.itertuples()]
    route_ids = frame.sample(frac=route_frac, random_state=seed)
    recs += [{"id": int(r.row_id), "task": "route", "intent": r.intent,
              "messages": [{"role": "system", "content": ROUTER_SYSTEM}, {"role": "user", "content": r.instruction},
                           {"role": "assistant", "content": route_target(r.intent, r.category)}]} for r in route_ids.itertuples()]
    rng = random.Random(seed); rng.shuffle(recs)
    return recs

def write_records(recs, path):
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

sft_train = sft_records(tr)
sft_val = sft_records(va.sample(500, random_state=SEED))
write_records(sft_train, SPLIT_DIR / "sft_train.jsonl"); write_records(sft_val, SPLIT_DIR / "sft_val.jsonl")
share = np.mean([r["task"] == "route" for r in sft_train])
print(f"sft_train: {len(sft_train)} records ({share:.1%} routing) | sft_val: {len(sft_val)}")

# re-check max_seq_length with the frozen prompts
texts = [tok.apply_chat_template(r["messages"], tokenize=False) for r in sft_train]
lens = np.array([len(x) for x in tok(texts, add_special_tokens=False)["input_ids"]])
MAX_SEQ = ceil32(np.percentile(lens, 98))
while (lens <= MAX_SEQ).mean() < 0.98:
    MAX_SEQ += 32
print(f"final max_seq_length = {MAX_SEQ} covers {(lens <= MAX_SEQ).mean():.2%} of SFT records (max {lens.max()})")
cfg = json.loads((D["config"] / "data_config.json").read_text())
cfg.update({"max_seq_length": MAX_SEQ, "system_prompt_sha256": PROMPT_HASH, "data_manifest_hash": MANIFEST_HASH})
(D["config"] / "data_config.json").write_text(json.dumps(cfg, indent=2))
(SPLIT_DIR / "sft_manifest.json").write_text(json.dumps(
    {"data_manifest_hash": MANIFEST_HASH, "system_prompt_sha256": PROMPT_HASH, "routing_share": float(share),
     "sft_train_sha256": sha256_file(SPLIT_DIR / "sft_train.jsonl"), "sft_val_sha256": sha256_file(SPLIT_DIR / "sft_val.jsonl"),
     "n_train_records": len(sft_train), "n_val_records": len(sft_val)}, indent=2))

for r in [next(x for x in sft_train if x["task"] == "response"), next(x for x in sft_train if x["task"] == "route"), sft_train[0]]:
    print("\n" + "=" * 70 + f"\n[{r['task']}] id={r['id']}\n" + tok.apply_chat_template(r["messages"], tokenize=False))

### Few-shot exemplars for B1 (five fixed pairs, one per category family)
Families: **orders** (ORDER, CANCEL) · **money** (REFUND, PAYMENT, INVOICE) · **account** (ACCOUNT, SUBSCRIPTION) · **logistics** (DELIVERY, SHIPPING) · **service** (CONTACT, FEEDBACK). Chosen from the *train* split with a fixed seed: clean basic-structure queries (tag `B`, no noise), short responses.

In [ ]:
FAMILY = {"ORDER": "orders", "CANCEL": "orders", "REFUND": "money", "PAYMENT": "money", "INVOICE": "money",
          "ACCOUNT": "account", "SUBSCRIPTION": "account", "DELIVERY": "logistics", "SHIPPING": "logistics",
          "CONTACT": "service", "FEEDBACK": "service"}
unknown = set(tr.category) - set(FAMILY)
print("categories not in the family map (assigned to 'service'):", unknown or "none")
tr_f = tr.assign(family=tr.category.map(lambda c: FAMILY.get(c, "service")),
                 words=tr.response.str.split().str.len())
cand = tr_f[tr_f["flags"].str.contains("B") & ~tr_f["flags"].str.contains("Z") & tr_f.words.between(30, 110)]
exemplars = []
for fam in ["orders", "money", "account", "logistics", "service"]:
    r = cand[cand.family == fam].sample(1, random_state=SEED).iloc[0]
    exemplars.append({"family": fam, "row_id": int(r.row_id), "intent": r.intent, "category": r.category,
                      "instruction": r.instruction, "response": r.response})
(SPLIT_DIR / "few_shot_exemplars.json").write_text(json.dumps(exemplars, indent=2, ensure_ascii=False))
for e in exemplars:
    print(f"[{e['family']:9s}] {e['intent']:25s} | {e['instruction'][:70]}")
EXEMPLAR_IDS = {e["row_id"] for e in exemplars}
assert not EXEMPLAR_IDS & set(mini.row_id) | EXEMPLAR_IDS & set(te.row_id), "exemplars must not be eval rows"

## 7. Baselines B0 and B1 (T18, T19)
* **B0** – base Qwen2.5-1.5B-Instruct (4-bit), frozen system prompt, zero-shot.
* **B1** – same prompt + the five exemplar pairs as chat turns (routing uses the same five rows as `instruction → JSON`).
* For each config and split: **response generation** (ROUGE-L, BERTScore-F1, leakage, perplexity) and **routing JSON** (intent accuracy / macro-F1, invalid-JSON rate).
* Greedy decoding, identical generation cap for every config. All predictions are cached on Drive; re-running resumes where it stopped.

In [ ]:
def run_config(config, split_name, frame, exemplars=None, bs_resp=GEN_BS_ZERO_SHOT, limit=None):
    sub = frame if limit is None else frame.sample(min(limit, len(frame)), random_state=SEED)
    sub = sub.sort_values("row_id").reset_index(drop=True)
    ids = sub["row_id"].tolist()
    base = f"{config}_{split_name}"
    resp_msgs = [build_response_messages(SYSTEM_PROMPT, ins, exemplars) for ins in sub.instruction]
    route_msgs = [build_route_messages(ROUTER_SYSTEM, ins, exemplars) for ins in sub.instruction]
    resp = generate_chat(model, tok, resp_msgs, ids, MAX_NEW_RESP, bs_resp, str(D["preds"] / f"{base}_response.jsonl"))
    route = generate_chat(model, tok, route_msgs, ids, MAX_NEW_ROUTE, GEN_BS_ROUTE, str(D["preds"] / f"{base}_route.jsonl"))

    outs = [r["output"] for r in resp]
    pred_intent = [parse_route_json(r["output"], ALL_INTENTS)[0] for r in route]
    im = intent_metrics(sub.intent, pred_intent, labels=sorted(set(sub.intent)))
    rl, rl_each = rouge_l(outs, sub.response.tolist())
    bf, bf_each = bertscore_f1(outs, sub.response.tolist())
    pd.DataFrame({"row_id": ids, "intent_gold": sub.intent, "intent_pred": pred_intent, "rougeL": rl_each,
                  "bertscore_f1": bf_each, "has_placeholder": [has_placeholder(o) for o in outs]}
                 ).to_csv(D["preds"] / f"{base}_scored.csv", index=False)
    row = {"run_id": base, "config": config, "split": split_name, "data_manifest_hash": MANIFEST_HASH,
           "prompt_hash": PROMPT_HASH[:12], "n": len(sub), "intent_acc": im["intent_acc"],
           "intent_macro_f1": im["intent_macro_f1"], "invalid_json_rate": im["invalid_rate"],
           "rougeL": rl, "bertscore_f1": bf, "leak_pct": placeholder_leakage(outs),
           "avg_resp_tokens": float(np.mean([r["n_new_tokens"] for r in resp])),
           "trunc_pct": 100 * float(np.mean([r["n_new_tokens"] >= MAX_NEW_RESP for r in resp]))}
    return row, sub

def add_perplexity_and_latency(row, sub, exemplars):
    ppl_items = [(build_response_messages(SYSTEM_PROMPT, i, exemplars), r) for i, r in
                 zip(sub.instruction[:PPL_N], sub.response[:PPL_N])]
    row["perplexity"] = response_perplexity(model, tok, ppl_items, batch_size=2 if exemplars else 4)
    if LATENCY_N:
        lat = latency_benchmark(model, tok, [build_response_messages(SYSTEM_PROMPT, i, exemplars) for i in sub.instruction[:LATENCY_N + 1]],
                                MAX_NEW_RESP, n=LATENCY_N)
        row.update(latency_stats(lat["seconds"])); row["tokens_per_s"] = lat["tokens_per_s"]; row["peak_gpu_gb"] = lat["peak_gpu_gb"]
    return row

In [ ]:
# ---- B0 zero-shot : eval-mini (re-uses the cached generations of the frozen variant) ----
for kind in ("response",):
    src = D["preds"] / f"promptcmp_{best}_evalmini_{kind}.jsonl"
    dst = D["preds"] / f"B0_evalmini_{kind}.jsonl"
    if src.exists() and not dst.exists():
        shutil.copy(src, dst)
row_b0_mini, _ = run_config("B0", "evalmini", mini)
log_row(row_b0_mini); print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row_b0_mini.items()})

In [ ]:
# ---- B0 zero-shot : test split (the long step; resumable) ----
row_b0_test, sub_test = run_config("B0", "test", te, limit=TEST_LIMIT)
row_b0_test = add_perplexity_and_latency(row_b0_test, sub_test, None)
log_row(row_b0_test); print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row_b0_test.items()})

In [ ]:
# ---- B1 few-shot : eval-mini and test ----
row_b1_mini, _ = run_config("B1", "evalmini", mini, exemplars, GEN_BS_FEW_SHOT)
log_row(row_b1_mini); print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row_b1_mini.items()})
row_b1_test, sub_test = run_config("B1", "test", te, exemplars, GEN_BS_FEW_SHOT, limit=TEST_LIMIT)
row_b1_test = add_perplexity_and_latency(row_b1_test, sub_test, exemplars)
log_row(row_b1_test); print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row_b1_test.items()})

In [ ]:
# ---- Optional: held-out-intent set (RQ5 reference for the base model) ----
if RUN_HELDOUT_BASELINES:
    for cfg_name, ex, bs in (("B0", None, GEN_BS_ZERO_SHOT), ("B1", exemplars, GEN_BS_FEW_SHOT)):
        r, _ = run_config(cfg_name, "heldout", ho, ex, bs, limit=TEST_LIMIT)
        log_row(r); print(cfg_name, "held-out:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()})
else:
    print("Skipped (RUN_HELDOUT_BASELINES=False).")

## 8. Summary — baseline table, figure and what is now on Drive

In [ ]:
res = pd.read_csv(RESULTS_PATH)
show = ["run_id", "n", "intent_acc", "intent_macro_f1", "invalid_json_rate", "rougeL", "bertscore_f1", "perplexity",
        "leak_pct", "avg_resp_tokens", "trunc_pct", "p50_s", "p95_s", "tokens_per_s", "peak_gpu_gb"]
display(res[show].round(4))

t = res[res.split == "test"].set_index("config")
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for a, col, title in zip(ax, ["intent_macro_f1", "bertscore_f1", "rougeL"], ["Intent macro-F1", "BERTScore-F1", "ROUGE-L"]):
    t[col].plot.bar(ax=a, color=["#9aa5b1", "#3b6ea5"][: len(t)], rot=0, title=title + " (test)")
    for p in a.patches:
        a.annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom")
plt.tight_layout(); plt.savefig(D["figures"] / "baselines_B0_B1_test.png", dpi=150); plt.show()

print("\nFrozen prompt:", best, "| manifest hash:", MANIFEST_HASH)
print("\nFiles now on Drive:")
for p in sorted(ROOT.rglob("*")):
    if p.is_file() and "hf_cache" not in p.parts:
        print(f"  {p.relative_to(ROOT)}  ({p.stat().st_size/1024:.0f} KB)")

## Done — what Notebook 02 needs from this one
`data/splits/*.jsonl` + `manifest.json` (hash-verified), `data/splits/sft_train.jsonl` / `sft_val.jsonl`, `prompts/system.txt` + `router_system.txt` + `frozen.json`, `config/data_config.json` (`max_seq_length`), `eval/*.py`, `results/results_log.csv` (B0/B1 rows) and `requirements.txt`.

Next: **Notebook 02 (QLoRA fine-tuning, C2 vs B0/B1)** loads these files, calls `verify_manifest()` and never recomputes the data.